# Urban Heat Island (UHI) Detection — Nairobi, Kenya

**Satellite:** Landsat 8 Collection 2 Tier 1 Level 2  
**Temperature Units:** Kelvin (analysis) / Celsius (display)  
**Analysis Period:** 2024-01-01 to 2024-12-31

This notebook detects and classifies Urban Heat Island intensity across Nairobi County
using Google Earth Engine (GEE) via the `earthengine-api` Python client. It also
overlays WorldPop 2020 population data to estimate heat-exposed residents.

---

## 1. Install & Authenticate Google Earth Engine


In [ ]:
# Install required packages (uncomment if needed)
# !pip install earthengine-api geemap

import ee

# Authenticate once — follow the browser prompt
# ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # <-- replace with your GEE project ID

print("Earth Engine initialised successfully.")


## 2. Define Region of Interest — Nairobi County (GAUL)

In [ ]:
kenya_regions = (
    ee.FeatureCollection('FAO/GAUL/2015/level1')
    .filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))
)

nairobi           = kenya_regions.filter(ee.Filter.stringContains('ADM1_NAME', 'Nairobi'))
analysis_boundary = nairobi.geometry()

print("Nairobi County boundary loaded.")


## 3. Define Analysis Period (Full Year 2024)

In [ ]:
start_date = '2024-01-01'
end_date   = '2024-12-31'


## 4. Load Dynamic World — Extract Urban / Built Pixels (Class 6)

Using Nairobi's dry seasons: **Jan–Feb** and **Jun–Aug**.  
Class 6 = Built / Urban area.


In [ ]:
dynamic_urban = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .select('label')
    .filterDate(start_date, end_date)
    .filterBounds(analysis_boundary)
    .filter(
        ee.Filter.Or(
            ee.Filter.calendarRange(1, 2, 'month'),   # Jan–Feb dry season
            ee.Filter.calendarRange(6, 8, 'month')    # Jun–Aug dry season
        )
    )
    .mode()
    .eq(6)
)

print("Urban/built pixel mask created (Dynamic World class 6).")


## 5. Inspect Landsat 8 Collection Metadata

In [ ]:
landsat8_raw = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .select('ST_B10')
    .filterBounds(analysis_boundary)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt('CLOUD_COVER', 10))
)

print('Landsat 8 collection size (cloud cover < 10 %):', landsat8_raw.size().getInfo())


## 6. Load Landsat 8 Thermal Band — Apply USGS C2 Scaling → Kelvin

**Conversion (USGS Collection 2 Level-2 constants):**

$$T_K = DN \times 0.00341802 + 149.0$$

Scale and offset are **hardcoded** from the official USGS product guide —
the metadata property keys do not exist in GEE and would return `null`.


In [ ]:
def scale_to_kelvin(image):
    lst_kelvin = (
        image
        .multiply(0.00341802)
        .add(149.0)
        .rename('LST_Kelvin')
    )
    return lst_kelvin.copyProperties(image, image.propertyNames())

landsat_thermal = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .select('ST_B10')
    .filterBounds(analysis_boundary)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt('CLOUD_COVER', 10))
    .map(scale_to_kelvin)
)

print("Scaled collection size:", landsat_thermal.size().getInfo())


## 7. Compute Median LST Composite (Kelvin & Celsius)

In [ ]:
median_thermal         = landsat_thermal.median()
median_thermal_celsius = median_thermal.subtract(273.15).rename('LST_Celsius')

print("Median composite computed.")
print("  Kelvin  band name:", median_thermal.bandNames().getInfo())
print("  Celsius band name:", median_thermal_celsius.bandNames().getInfo())


## 8. Compute Mean LST Across Nairobi (UHI Baseline)

In [ ]:
mean_lst_k = ee.Number(
    median_thermal.reduceRegion(
        reducer   = ee.Reducer.mean(),
        geometry  = analysis_boundary,
        scale     = 100,
        maxPixels = int(1e13)
    ).values().get(0)
)

mean_lst_c = mean_lst_k.subtract(273.15)

print(f"Mean LST — Kelvin:  {mean_lst_k.getInfo():.4f} K")
print(f"Mean LST — Celsius: {mean_lst_c.getInfo():.4f} °C")


## 9. Calculate UHI Index

$$\text{UHI Index} = \frac{T_i - \bar{T}}{\bar{T}}$$

- **Positive** → warmer than city average (heat island)  
- **Negative** → cooler than city average (cool island)


In [ ]:
uhi_index = median_thermal.expression(
    '(TIR - MEAN) / MEAN',
    {
        'TIR' : median_thermal,
        'MEAN': mean_lst_k
    }
).rename('UHI_Index')

# Inspect actual value range to verify threshold choices
uhi_range = uhi_index.reduceRegion(
    reducer   = ee.Reducer.minMax(),
    geometry  = analysis_boundary,
    scale     = 100,
    maxPixels = int(1e13)
).getInfo()

print("UHI Index value range:")
print(f"  Min: {uhi_range.get('UHI_Index_min', 'N/A'):.6f}")
print(f"  Max: {uhi_range.get('UHI_Index_max', 'N/A'):.6f}")


## 10. Classify UHI Intensity into 5 Categories

Only pixels **warmer than the city mean** (positive UHI index) within
**built-up areas** are classified.

| Class | UHI Index Range | Interpretation |
|---|---|---|
| 1 | 0.000 – 0.005 | Mild |
| 2 | 0.005 – 0.010 | Moderate |
| 3 | 0.010 – 0.015 | Strong |
| 4 | 0.015 – 0.020 | Very Strong |
| 5 | ≥ 0.020 | Extreme |


In [ ]:
uhi_positive = uhi_index.gte(0)   # Only classify pixels warmer than mean

uhi_classes = (
    ee.Image.constant(0).byte()
    .where(uhi_index.gte(0.000).And(uhi_index.lt(0.005)), 1)   # Mild
    .where(uhi_index.gte(0.005).And(uhi_index.lt(0.010)), 2)   # Moderate
    .where(uhi_index.gte(0.010).And(uhi_index.lt(0.015)), 3)   # Strong
    .where(uhi_index.gte(0.015).And(uhi_index.lt(0.020)), 4)   # Very Strong
    .where(uhi_index.gte(0.020), 5)                             # Extreme
    .updateMask(dynamic_urban)    # Restrict to built-up areas only
    .updateMask(uhi_positive)     # Exclude cool pixels (negative UHI index)
)

print("UHI classification complete.")


## 11. Load WorldPop 2020 Population Data

In [ ]:
world_pop = (
    ee.ImageCollection("WorldPop/GP/100m/pop")
    .filter(ee.Filter.eq('country', 'KEN'))
    .filterDate('2020-01-01', '2020-12-31')
    .first()
    .clip(analysis_boundary)
)

print("WorldPop 2020 image loaded.")


## 12. Estimate Population Exposed to High Heat (UHI Class ≥ 4)

**High Heat threshold:** UHI class 4 (Very Strong) or 5 (Extreme).


In [ ]:
high_heat_mask  = uhi_classes.gte(4)
vulnerable_pop  = world_pop.updateMask(high_heat_mask)

total_vulnerable = vulnerable_pop.reduceRegion(
    reducer   = ee.Reducer.sum(),
    geometry  = analysis_boundary,
    scale     = 100,
    maxPixels = int(1e13)
).getInfo()

print("Population in High UHI Zones (class ≥ 4):")
print(f"  {list(total_vulnerable.values())[0]:,.0f} people")


## 13. Population Breakdown by UHI Class

In [ ]:
import pandas as pd

pop_uhi = world_pop.addBands(uhi_classes)

pop_by_class = pop_uhi.reduceRegion(
    reducer = ee.Reducer.sum().group(
        groupField = 1,
        groupName  = 'UHI_Class'
    ),
    geometry  = analysis_boundary,
    scale     = 100,
    maxPixels = int(1e13)
).getInfo()

groups = pop_by_class.get('groups', [])
df_pop = pd.DataFrame([
    {'UHI_Class': g['UHI_Class'], 'Population': g['sum']}
    for g in groups
]).sort_values('UHI_Class').reset_index(drop=True)

label_map = {1: 'Mild', 2: 'Moderate', 3: 'Strong', 4: 'Very Strong', 5: 'Extreme'}
df_pop['Intensity'] = df_pop['UHI_Class'].map(label_map)
df_pop['Population'] = df_pop['Population'].round(0).astype(int)

print("=== Population by UHI Class ===")
display(df_pop[['UHI_Class', 'Intensity', 'Population']])


## 14. Visualise Population Distribution by UHI Class

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

colors = ['#add8e6', '#ffff00', '#ffa500', '#a52a2a', '#ff0000']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Column chart
axes[0].bar(df_pop['UHI_Class'], df_pop['Population'], color=colors[:len(df_pop)])
axes[0].set_title('Population by UHI Class — Bar Chart', fontweight='bold')
axes[0].set_xlabel('UHI Class (1 = Mild → 5 = Extreme)')
axes[0].set_ylabel('Population')
axes[0].set_xticks(df_pop['UHI_Class'])
axes[0].set_xticklabels([f"{r['UHI_Class']}\n({r['Intensity']})" for _, r in df_pop.iterrows()])
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Line chart
axes[1].plot(df_pop['UHI_Class'], df_pop['Population'],
             marker='o', linewidth=3, markersize=8, color='#d62728')
axes[1].set_title('Population by UHI Class — Line Chart', fontweight='bold')
axes[1].set_xlabel('UHI Class (1 = Mild → 5 = Extreme)')
axes[1].set_ylabel('Population')
axes[1].set_xticks(df_pop['UHI_Class'])
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle('Population Exposure to Urban Heat Island Intensity — Nairobi 2024',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('uhi_population_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as uhi_population_chart.png")


## 15. High vs. Low Heat Population Summary

In [ ]:
high_heat = uhi_classes.gte(3)
low_heat  = uhi_classes.lt(3)

def pop_sum(mask):
    result = world_pop.updateMask(mask).reduceRegion(
        reducer   = ee.Reducer.sum(),
        geometry  = analysis_boundary,
        scale     = 100,
        maxPixels = int(1e13)
    ).getInfo()
    return list(result.values())[0]

high_pop  = pop_sum(high_heat)
low_pop   = pop_sum(low_heat)
total_pop = pop_sum(ee.Image.constant(1))   # no mask = full count

print(f"{'Category':<25} {'Population':>15}")
print("-" * 42)
print(f"{'High Heat (class ≥ 3)':<25} {high_pop:>15,.0f}")
print(f"{'Low Heat  (class < 3)':<25} {low_pop:>15,.0f}")
print(f"{'Total Population':<25} {total_pop:>15,.0f}")
print(f"\nHigh-heat share: {high_pop / total_pop * 100:.1f}%")


## 16. Interactive Map Visualisation

Uses [`geemap`](https://geemap.org) for interactive GEE layer display.


In [ ]:
import geemap

Map = geemap.Map()
Map.centerObject(analysis_boundary, 10)

Map.addLayer(analysis_boundary, {'color': 'yellow'}, 'Nairobi County Boundary')

Map.addLayer(
    dynamic_urban.clip(analysis_boundary),
    {'min': 0, 'max': 1, 'palette': ['white', 'grey']},
    'Urban Mask', False
)

Map.addLayer(
    median_thermal.clip(analysis_boundary),
    {'min': 295, 'max': 315, 'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']},
    'Median LST (Kelvin)', False
)

Map.addLayer(
    median_thermal_celsius.clip(analysis_boundary),
    {'min': 22, 'max': 42, 'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']},
    'Median LST (Celsius)', False
)

Map.addLayer(
    uhi_index.clip(analysis_boundary),
    {'min': -0.01, 'max': 0.025, 'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']},
    'UHI Index (raw)', False
)

Map.addLayer(
    uhi_classes.clip(analysis_boundary),
    {'min': 1, 'max': 5, 'palette': ['lightblue', 'yellow', 'orange', 'brown', 'red']},
    'UHI Classes (Urban Only)', True
)

Map.addLayer(
    world_pop,
    {'min': 0, 'max': 500, 'palette': ['white', 'yellow', 'orange', 'red']},
    'WorldPop 2020', False
)

Map.addLayer(
    vulnerable_pop,
    {'min': 0, 'max': 500, 'palette': ['white', 'orange', 'red']},
    'Population in High UHI Zones', False
)

Map


## 17. Export All Outputs to Google Drive

In [ ]:
export_base = dict(
    region    = analysis_boundary,
    scale     = 100,
    crs       = 'EPSG:4326',
    maxPixels = int(1e13),
    folder    = 'UrbanHeat'
)

# A) Vulnerable population raster
task_a = ee.batch.Export.image.toDrive(
    image       = vulnerable_pop,
    description = 'Population_High_UHI_Nairobi_2024',
    **export_base
)
task_a.start()
print("Task A — Vulnerable population raster:", task_a.status())

# B) UHI Classified Raster
task_b = ee.batch.Export.image.toDrive(
    image       = uhi_classes.clip(analysis_boundary),
    description = 'UHI_Classes_Nairobi_Landsat8_2024',
    **export_base
)
task_b.start()
print("Task B — UHI classes raster:", task_b.status())

# C) Raw UHI Index (urban pixels only)
task_c = ee.batch.Export.image.toDrive(
    image       = uhi_index.clip(analysis_boundary).updateMask(dynamic_urban),
    description = 'UHI_Index_Raw_Nairobi_Landsat8_2024',
    **export_base
)
task_c.start()
print("Task C — Raw UHI index raster:", task_c.status())
